In [378]:
import numpy as np
import pandas as pd

class topo2rest():
   def __init__(self, ifile:str, temps:list = [300.0, 500.0], nreps:int = 20, kappa:float = 1.00):
      '''Convert processed topology to REST2/3 input topology
         E_tot = gamma*E^{pp} + sqrt(gamma)*E^{pw} + E^{ww}
         gamma = T_0/T_i
         for REST2/3 bonds and angles are not scaled per the REST2 paper,
         however, for REST2 LJ parameters epsilon_i is scaled by epsilon_i*gamma
         for tempered atoms and all others are unmodified. 
         In the case of REST3, with the additon of sqrt(gamma)*kappa*E^{pw} which differs 
         from sqrt(gamma)*E^{pw}, we produced the combination rule 2 nonbonded terms
         involving protein-water interactions to override the pw nonbonded interactions
         as including gamma*kappa with each hot atom epsilon_i would result in the 
         incorrect form of: E_tot = gamma*kappa*E^{pp} + sqrt(gamma*kappa)*E^{pw} + E^{ww}:
         rather than the correct form: 
         E_tot = gamma*kappa*E^{pp} + sqrt(gamma)*kappa*E^{pw} + E^{ww}
         input
         ifile = inputtopology.top
         temps = ["lower temp":float, "upper temp":float ]; temperature range of replicas
         kappa:float = kappa scaling; if not equal to 1 REST3 implementation active
         hot_m = hot molecule; [0] for most systems will select the protein'''

      self.nreps = nreps
      self.kappa = kappa
      self.hot_m = None
      self.tempreps = self.compute_temperatures(temps)
      self.lambdai = self.compute_lambda()
      self.sections = {}
      self.sections_out = {}
      self.scaled_dihedrals = {}
      self.scaled_dihedral_types = {}
      self.hard_order_sections = [ "defaults", "atomtypes", "nonbond_params", "bondtypes", \
                                   "constrainttypes", "angletypes", "dihedraltypes" , "moleculetype"]
      with open(ifile) as topo:
         self.readfile = topo.readlines()
      self.gather_param_sections()
         
   def compute_lambda(self):
      return [ Ti/self.tempreps[0] for Ti in self.tempreps ]
   
   def compute_temperatures(self, temp_range:list):
      from numpy import log, exp
      tlow, thigh = temp_range
      temps = []
      for i in range(self.nreps):
         temps.append(tlow*exp((i)*log(thigh/tlow)/(self.nreps-1)))
      return temps
   
   def parse_section(self,trunks):
      first_round = True
      output = []
      for line in self.readfile[trunks:]:
         if "[" not in line and first_round!=True:
            output.append(line)
         elif ";" == line[0]:
            #output.append(line)
            continue
         elif "[" in line and first_round!=True: 
            break
         first_round = False
      return output

   def get_molecule_atomtypes(self,molecule:int=0):
      import pandas as pd
      b=[]
      for i in self.sections['moleculetype'][molecule]['atoms']:
         if len(i.split())>1 and ';' not in i.split()[0] and '[' not in i.split()[0] :
            b.append(i.split()[:2])
      dataset = np.array(b,dtype=object)
      print(dataset)   
      # Need to grab unique atoms
      atomtypes = dataset[:,1]
      return np.unique(atomtypes)
   
   def get_scale_nonbonded(self):
      import pandas as pd
      for i in self.hot_m:
         atom_es = self.get_molecule_atomtypes()
      # TODO everything
   
#   def show_molecule_names(self):
#      try:
#         pass
         

   def get_molecule_atoms(self):
      import pandas as pd
      b=[]
      for i in self.sections['atoms']:
         i.split()
         if len(i.split())>1 and i.split()[0]!=';' and i.split()[0]!='[' :
            b.append(i.split())
   
   def _moleculetype_sub(self,linestart:int):
      first_round = True
      output = []
      for line in self.readfile[linestart:]:
         if "[" not in line and ';' not in line[:3]:
            output.append(line)
         elif ";" == line[0]:
            #output.append(line)
            continue
         elif "[" in line and first_round!=True: 
            break
         first_round = False
      return output 
   
   def identify_moltype_sections(self,trunks:int):
      section_start = []
      for i, line in enumerate(self.readfile[trunks:]):
         if '[' in line and 'moleculetype' not in line and 'system' not in line:
            section_start.append(i+trunks)
         elif i != 0 and 'moleculetype' in line or 'system' in line:
            break
      return section_start

   def parse_moleculetypes(self,trunks:int):
      first_round = True
      output = {}
      sections = self.identify_moltype_sections(trunks)
      output['header'] = self.readfile[trunks:trunks+2]
      for section in sections:
         section_ = self.readfile[section].split()[1]
         output[section_] = self._moleculetype_sub(section)
      return output
   
   def get_scale_dehedrals_(self, hot_m:list = [0]):
      # need to get atom reference for which parameters to scale 
#      dihedraltypes = pd.DataFrame([i.split()[:-2] if i.split()[-2] == ';' else i.split()[:-1] if i.split()[-1] == ';' \
#                               else i.split() for i in self.sections['dihedraltypes'] if ';' not in i[:3] if '[' not in i[:3] \
#                               if '\n' not in i[:3]], columns=['i','j','k','l','func','phase','K','mult'])
#      print(dihedraltypes)
      
      dihedrals_new = {}
      dihedral_types_new = {}
      dihedrals = self.sections['moleculetype'][hot]['dihedrals']
      dihedral_types = [" ".join(i.split()[:-2]) if i.split()[-2] == ';' else i.split()[:-1] if i.split()[-1] == ';' \
                               else i.split() for i in self.sections['dihedraltypes'] if ';' not in i[:3] if '[' not in i[:3] \
                               if '\n' not in i[:3]]
      for hot in hot_m:
         for lambdai in self.lambdai:
            dih_new = []
            dih_types_new = []
            for dihedral in dihedrals:
               dls_ = dihedral.split()
               if len(dls_) == 5:
                  dih_new.append(dihedral)
               elif len(dls_) == 8:
                  Kscaled = float(dls_[6])*lambdai
                  stringout = f'{dls_[0]:>5d} {dls_[1]:>5d} {dls_[2]:>5d} {dls_[3]:>5d} {dls_[4]:^9d}{dls_[6]:<10.5f}{Kscaled:<10.3f}{dls_[7]}\n'
                  dih_new.append(stringout)
               else: print("Warning: incorrect parsing of dihedrals section\n expecting 5 or 8 columns\n{line}")
            for dihedraltype in dihedral_types:
               dtls_ = dihedraltype.split()
               Kscaled = float(dtls_[6])*lambdai
               if len(dtls_) == 8:
                  dih_types_new.append(dihedraltype)
                  if dtls_[0] !='X' and dtls_[3] != 'X':
                     stringout = f'{"s"+dtls_[0]:>5d} {"s"+dtls_[1]:>5d} {"s"+dtls_[2]:>5d} {"s"+dtls_[3]:>5d} {dtls_[4]:^9d}{dtls_[6]:<10.5f}{Kscaled:<10.3f}{dtls_[7]}\n'
                     dih_types_new.append(stringout)
                  elif dtls_[3] == 'X':
                     if dtls_[0] == 'X':
                        stringout = f'{dtls_[0]:>5d} {"s"+dtls_[1]:>5d} {"s"+dtls_[2]:>5d} {dtls_[3]:>5d} {dtls_[4]:^9d}{Kscaled:<10.3f}{dtls_[6]:<10.5f}{dtls_[7]}\n' 
                        dih_types_new.append(stringout)
                  else: print(f'warning: parameter not found for {dtls_[:4]}')
               else: print('warning: dihedraltype not processed:\n '+dihedraltype)
                     
            dihedrals_new[lambdai] = dih_new
            dihedral_types_new[lambdai] = dih_types_new
            
         self.scaled_dihedrals[hot] = dihedrals_new
         self.scaled_dihedral_types[hot] = dihedral_types_new

   def gather_param_sections(self):
      for section in self.hard_order_sections[:-1]:
         is_select = [ i for i, line in enumerate(self.readfile) if section in line ]
         in_select = [f' [ {section} ] \n']
         for i in is_select:
            in_select += self.parse_section(i)
         self.sections[section]=in_select
      section = self.hard_order_sections[-1]
      is_select = [ i for i, line in enumerate(self.readfile) if section in line ] 
      moltype_dict = {} 
      for i in range(len(is_select)):
         moltype_dict[i] = self.parse_moleculetypes(is_select[i])
      self.sections[section] = moltype_dict
         

In [379]:
test = topo2rest('./example_topo/processed.top')

In [381]:
test.get_molecule_atomtypes()

[['1' 'N3']
 ['2' 'H']
 ['3' 'H']
 ...
 ['622' 'C']
 ['623' 'O2']
 ['624' 'O2']]


array(['C', 'C1', 'C5', 'C6', 'C7', 'C8', 'C9', 'CA', 'CT', 'H', 'H1',
       'HA', 'HB', 'HC', 'HO', 'HP', 'N', 'N3', 'O', 'O2', 'O3', 'OB',
       'OH', 'S'], dtype=object)

In [373]:
var_floatA, var_floatB, var_int = 180.8, 2.71960, 5 
print(f'{var_int:>5d} {var_int:>5d} {var_int:>5d} {var_int:>5d} {var_int:^9d}{var_floatA:<10.3f}{var_floatB:<10.5f}{var_int}\n')

    5     5     5     5     5    180.800   2.71960   5



In [374]:
test.sections['dihedraltypes']

[' [ dihedraltypes ] \n',
 ';i  j   k  l\t func      phase      kd      pn\n',
 ' C   C1  N   O     4     180.0      4.60240     2  ; Amber99Sb-disp\n',
 ' C   C1  N   OB    4     180.0      4.60240     2  ; Amber99Sb-disp\n',
 ' C   C1  N   H     4     180.0      4.60240     2  ; Amber99Sb-disp\n',
 ' C   C1  N   HB    4     180.0      4.60240     2  ; Amber99Sb-disp\n',
 ' C9  O   C   OH    4     180.0     43.93200     2  ; Amber99Sb-disp\n',
 ' CB  CK  N*  CT    4     180.0      4.18400     2  ; Amber99Sb-disp\n',
 ' C   CM  N*  CT    4     180.0      4.18400     2  ; Amber99Sb-disp\n',
 ' CT  O   C   OH    4     180.0     43.93200     2  ; Amber99Sb-disp\n',
 ' CT  CV  CC  NA    4     180.0      4.60240     2  ; Amber99Sb-disp\n',
 ' CT  CW  CC  NB    4     180.0      4.60240     2  ; Amber99Sb-disp\n',
 ' CT  CC  CW  NB    4     180.0      4.60240     2  ; Amber99Sb-disp\n',
 ' CT  CW  CC  NA    4     180.0      4.60240     2  ; Amber99Sb-disp\n',
 ' CB  CT  C*  CW    4     180.0 

In [293]:
test_dihedral_df = pd.DataFrame([i.split() for i in test.sections['moleculetype'][0]['dihedrals']],columns=['i','j','k','l','type','phase','K','period']).astype({'i':int,'j':int,)

In [336]:
if not test.sections['moleculetype'][0]['atoms'][-1].split():
    print("Yes")

Yes


In [359]:
test_atoms_df = pd.DataFrame(np.array(
    [i.split()[:2] for i in test.sections['moleculetype'][0]['atoms'] if len(i.split()) != 0 ]),
    columns=['number','name']).astype({"name":str, "number":int})

In [361]:
test_atoms_df.dtypes

number     int64
name      object
dtype: object

In [360]:
for i,j,k,l in test_dihedral_df[['i','j','k','l']].values:
    if i != None and j != None and k != None and l != None:
        a, b, c, d = None, None, None, None
        for atmname, atm in zip([a, b, c, d],[i, j, k, l]):
            print(atm)
            atmname = test_atoms_df[test_atoms_df['number'] == atm]
        print(a,b,c,d)

5
10
8
9
None None None None
8
12
10
11
None None None None
8
10
12
25
None None None None
10
12
25
27
None None None None
10
12
25
27
None None None None
10
12
25
27
None None None None
10
12
25
27
None None None None
10
12
25
27
None None None None
10
12
25
27
None None None None
12
27
25
26
None None None None
25
29
27
28
None None None None
25
27
29
35
None None None None
27
29
35
37
None None None None
27
29
35
37
None None None None
27
29
35
37
None None None None
27
29
35
37
None None None None
27
29
35
37
None None None None
27
29
35
37
None None None None
29
37
35
36
None None None None
35
39
37
38
None None None None
35
37
39
46
None None None None
37
39
46
48
None None None None
37
39
46
48
None None None None
37
39
46
48
None None None None
37
39
46
48
None None None None
37
39
46
48
None None None None
37
39
46
48
None None None None
39
48
46
47
None None None None
46
50
48
49
None None None None
46
48
50
60
None None None None
48
50
60
62
None None None None
48
50
60
62
N

In [324]:
test_atoms_df['number']==1

0      False
1      False
2      False
3      False
4      False
       ...  
620    False
621    False
622    False
623    False
624    False
Name: number, Length: 625, dtype: bool

In [295]:
test.sections['moleculetype'][0]['atoms']

['     1         N3      1    GLY      N      1     0.2943      14.01\n',
 '     2          H      1    GLY     H1      2     0.1642      1.008\n',
 '     3          H      1    GLY     H2      3     0.1642      1.008\n',
 '     4          H      1    GLY     H3      4     0.1642      1.008\n',
 '     5         C1      1    GLY     CA      5      -0.01      12.01\n',
 '     6         HP      1    GLY    HA1      6     0.0895      1.008\n',
 '     7         HP      1    GLY    HA2      7     0.0895      1.008\n',
 '     8          C      1    GLY      C      8     0.6163      12.01\n',
 '     9         OB      1    GLY      O      9    -0.5722         16   ; qtot 1\n',
 '    10          N      2    MET      N     10    -0.4157      14.01\n',
 '    11         HB      2    MET      H     11     0.2719      1.008\n',
 '    12         CT      2    MET     CA     12    -0.0237      12.01\n',
 '    13         H1      2    MET     HA     13      0.088      1.008\n',
 '    14         CT      2 

In [294]:
test_dihedral_df

,i,j,k,l,type,phase,K,period
0,5,10,8,9,4,None,None,None
1,8,12,10,11,4,None,None,None
2,8,10,12,25,4,90.0,1.67360,2
3,10,12,25,27,4,160.888,2.71960,1
4,10,12,25,27,4,90.0,-1.0669,1
...,...,...,...,...,...,...,...,...
390,609,623,622,624,4,None,None,None
391,614,619,617,618,4,None,None,None
392,617,620,619,621,4,None,None,None
393,None,None,None,None,None,None,None,None


In [157]:
f=open('test_dihedraltypes.txt','w')
f.writelines(test.sections['dihedraltypes'])
f.close()
